In [1]:
from src.utils import get_data_env
from src.models import SpatialGNNModel
from src.dataloading import DeepDataLoader

/home/de-schrijvers/.conda/envs/gnenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from src.models.deep.gru import GRUModel

In [3]:
disease_name    = 'influenza'
nuts_level      = 'nuts3'
min_date        ='2006-05-15'
max_date        = '2020-06-01'
split_trainval  = '2018-06-01'
split_valtest   = '2019-06-01'
split_berlin    = False


horizon_size    = 1
horizon_leadtime= 3
sequence_length = 1
lags            = 1

graphtype       = 'boolean_neighbors_self'
modelname       = 'boolean neighbors - spatial gcn'

# training hparams
n_epochs        = 100
lr              = 0.0005
min_delta       = 0
loss            = 'mse'

global_hparams = {
    "lr"                : lr,
    "n_epochs"          : n_epochs,
    "scheduler"         : 'plateau',
    "scheduler_kwargs"  :   {'mode': 'min', 'factor': 0.5, 'patience': 7},
    'min_delta'         : min_delta,
    'loss'              : loss,
    'patience'          : 20,
    }


In [4]:

logged_zscore = DeepDataLoader(disease_name, get_data_env(), nuts_level=nuts_level, min_date=min_date,max_date=max_date, include_population=False, horizon_size = horizon_size, horizon_leadtime = horizon_leadtime, sequence_length=sequence_length, split_berlin=split_berlin)
logged_zscore.add_time_features()
logged_zscore.log_transform_target()
logged_zscore.set_splits(split_trainval, split_valtest)
logged_zscore.normalize()
logged_zscore.add_lagged_features(lags = lags)
logged_zscore.finalize()

dl1 = logged_zscore.copy().retrieve_graph('boolean_neighbors_self').construct_dataloaders()


Dataloader temporal windowing: extending data collection from 2006-05-15 to 2006-04-17 (+4 weeks)
berlin districts removed


In [5]:
gru = GRUModel(dl1)
gru.set_model_hparams()
gru.set_global_hparams()
gru.train()

Dataloader Snapshot: GraphDataLoaderEntry(x=(400, 3, 1), y=(400, 1), edge_index=(2, 2488), edge_weight=(2488,))

==          Training unknown          ==


AttributeError: 'tuple' object has no attribute 'dim'